# Prophet baseline

A direct forecast over the final 15% of the hourly series. Unlike the lagged XGBoost and LSTM experiments, this model does not observe targets inside the test period. The original notebook used the old `fbprophet` package name; this version uses `prophet`.


In [ ]:
from prophet import Prophet


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [Path("../data/clean_hourly_data.csv"), Path("data/clean_hourly_data.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("The reconstructed CSV is missing; see data/README.md")

data = pd.read_csv(data_path, parse_dates=["ds"])
required = {"ds", "y"}
if not required.issubset(data.columns):
    raise ValueError("The CSV must contain ds (timestamp) and y (numeric target) columns")
data = data[["ds", "y"]].copy().sort_values("ds").reset_index(drop=True)
data["y"] = pd.to_numeric(data["y"], errors="raise")
if data.isna().any().any() or data["ds"].duplicated().any():
    raise ValueError("Remove missing values and duplicate timestamps before running these notebooks")
if not data["ds"].diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("These notebooks require consecutive hourly observations")

train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)
if train_end < 48 or validation_end - train_end < 24 or len(data) - validation_end < 24:
    raise ValueError("The dataset is too short for the chronological split")
print(f"Rows: {len(data):,}; train: {train_end:,}; validation: {validation_end - train_end:,}; test: {len(data) - validation_end:,}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def report(y_true, y_pred, timestamps, title):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{title} | MAE: {mae:.3f} | RMSE: {rmse:.3f}")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(timestamps, y_true, label="Observed", linewidth=1)
    ax.plot(timestamps, y_pred, label="Predicted", linewidth=1)
    ax.set(title=title, xlabel="Time", ylabel="Hourly sum of minute-level kW")
    ax.legend()
    fig.tight_layout()
    plt.show()


## Fit the baseline and forecast the test timestamps


In [ ]:
train_and_validation = data.iloc[:validation_end].copy()
test = data.iloc[validation_end:].copy()

model = Prophet(interval_width=0.95, daily_seasonality=True, weekly_seasonality=True)
model.fit(train_and_validation)
forecast = model.predict(test[["ds"]])
report(test["y"].to_numpy(), forecast["yhat"].to_numpy(),
       test["ds"], "Prophet: direct test-horizon forecast")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test["ds"], test["y"], label="Observed", linewidth=1)
ax.plot(test["ds"], forecast["yhat"], label="Forecast", linewidth=1)
ax.fill_between(test["ds"], forecast["yhat_lower"], forecast["yhat_upper"], alpha=0.2, label="95% interval")
ax.set(xlabel="Time", ylabel="Hourly sum of minute-level kW", title="Prophet forecast interval")
ax.legend()
fig.tight_layout()
plt.show()


**Interpretation:** The source is one household in Sceaux, France. No holiday calendar is added in this baseline, so seasonality can be examined without holiday assumptions.
